In [95]:
# Install and load necessary packages
# install.packages(c("ggpubr"))
library(vegan)
library(phyloseq)
library(ggplot2)
library(ape)
library(picante)
library(pheatmap)
library(ggpubr)
library(zCompositions)

# Load data
asv_table_str = "/Users/theco/Library/CloudStorage/Dropbox/Projects/niche_neutral/otu_table_treeholes_dada2_renamed_clean_red_phyloseq.tsv"
sample_table_str <- "../../1_samples_properties/Metadata_singleTree_wDirich.tsv"  
taxonomy_str <- "../../1_dada2/taxonomy.list"

# --- Minimum number of reads to keep a sample/taxa
Nprune.sample = 1000
Nprune.taxa = 10

## Preprocess otu_table

In [88]:
# Load OTU table
otu_table <- read.csv(asv_table_str, header = TRUE, sep = "\t", row.names = 1)

# Remove any text columns from otu_table
otu_table <- otu_table[, sapply(otu_table, is.numeric)]

# Convert remaining columns to numeric if needed
# otu_table <- as.data.frame(lapply(otu_table, as.numeric))
otu_table

,F.D.DNA.P1.A3.A.01.ID3.3.15.THS.2.16S.3,F.D.DNA.P1.A3.A.02.ID3.3.15.THS.10.16S.3,F.D.DNA.P1.A3.A.03.ID10.3.15.THS.9.16S.3,F.D.DNA.P1.A3.A.04.ID7.7.15.THS.6.16S.3,F.D.DNA.P1.A3.A.05.ID10.2.15.THS.4.16S.3,F.D.DNA.P1.A3.A.06.ID17.2.15.THS.1.16S.3,F.D.DNA.P1.A3.A.09.ID03.11.15.THS.6.16S.3,F.D.DNA.P1.A3.A.10.ID29.09.15.THS.4.16S.3,F.D.DNA.P1.A3.A.11.ID14.03.15.THS.1.16S.3,F.D.DNA.P1.A3.A.12.ID11.05.15.THS.1.16S.3,...,F.D.DNA.P3.C3.F.03.ID16.06.15.THS.3.16S.3,F.D.DNA.P3.C3.F.05.ID23.06.15.THS.9.16S.3,F.D.DNA.P3.C3.G.01.ID09.06.15.THS.7.16S.3,F.D.DNA.P3.C3.G.02.ID14.01.15.THS.6.16S.3,F.D.DNA.P3.C3.G.03.ID16.06.15.THS.4.16S.3,F.D.DNA.P3.C3.G.04.ID23.06.15.THS.1.16S.3,F.D.DNA.P3.C3.G.05.ID23.06.15.THS.10.16S.3,F.D.DNA.P3.C3.H.02.ID14.01.15.THS.7.16S.3,F.D.DNA.P3.C3.H.03.ID16.06.15.THS.5.16S.3,F.D.DNA.P3.C3.H.04.ID23.06.15.THS.3.16S.3
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,...,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
ASV_1,170,4,371,213,66,63,35,100,115,104,...,18,338,177,51,228,331,15,214,0,15
ASV_10,0,30,0,0,1,0,0,0,0,0,...,240,0,0,0,2,0,7,0,1,319
ASV_100,13,0,17,23,3,7,0,19,12,45,...,4,24,160,0,18,14,0,9,0,1
ASV_1000,2,0,8,0,1,1,0,0,3,1,...,0,2,17,0,0,1,0,20,0,0
ASV_1001,1,0,0,8,1,4,3,3,1,6,...,1,4,4,1,1,2,0,6,0,0
ASV_1002,0,0,1,0,1,3,0,1,1,3,...,0,0,29,0,0,0,0,13,0,0
ASV_1003,9,0,0,1,6,1,1,1,0,0,...,0,0,0,4,4,4,0,0,0,0
ASV_1004,3,1,2,0,0,0,0,1,0,0,...,1,0,0,3,2,4,4,1,2,3
ASV_1005,4,1,0,0,5,0,0,8,0,0,...,2,1,0,0,2,1,1,0,0,0


In [91]:
# Load OTU table
otu_table <- read.csv(asv_table_str, header = TRUE, sep = "\t", row.names = 1)

# Remove any text columns from otu_table
otu_table <- otu_table[, sapply(otu_table, is.numeric)]

otu_table <- otu_table(as.matrix(otu_table), taxa_are_rows = TRUE)

# Load sample data
sample_data <- read.table(sample_table_str, header = TRUE, sep = "\t", row.names = 1)
sample_data <- sample_data(sample_data)

# Load taxonomy data
tax_table <- read.table(taxonomy_str, sep = ";", header = FALSE, skip = 1)
colnames(tax_table) <- c("Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species")
rownames(tax_table) <- tax_table$Species
tax_table <- tax_table(as.matrix(tax_table))

dim(otu_table)
dim(env_data)
dim(tax_table)

[1] 3540  165

[1] 165  33

[1] 8651    7

In [96]:
# Create phyloseq object
physeq <- phyloseq(otu_table, sample_data, tax_table)

# Prune samples and taxa
physeq <- prune_samples(sample_sums(physeq) >= Nprune.sample, physeq)
physeq <- prune_taxa(taxa_sums(physeq) >= Nprune.taxa, physeq)

# Extract pruned OTU table and sample data
otu_table <- as.data.frame(otu_table(physeq))
env_data <- as.data.frame(sample_data(physeq))

print(dim(otu_table))
print(dim(env_data))


[1] 3540  165
[1] 165  33


In [101]:
# Step 2: Prepare Data

otu_table_pseudo <- cmultRepl(otu_table, 
                              method = "GBM", 
                              output = "prop",
                              z.warning = 0.95,  # Increase the warning threshold
                              z.delete = 1,      # Prevent deletion
                              suppress.print = TRUE)
dim(otu_table_pseudo)

clr_transformed <- decostand(otu_table_pseudo, method = "clr")
dim(clr_transformed)

# Transpose the CLR-transformed data so that samples are columns and ASVs are rows
clr_transformed_t <- t(clr_transformed)

# Compute Aitchison distance (Euclidean distance on transposed CLR-transformed data)
aitchison_dist <- dist(clr_transformed_t)


[1] 3540  165

In [121]:
# Perform PCoA
pcoa_results <- cmdscale(aitchison_dist, eig = TRUE, k = 2) # k = number of dimensions
scores <- as.data.frame(pcoa_results$points) # Convert PCoA results to a data frame
colnames(scores) <- c("PCoA1", "PCoA2") # Rename columns for better readability

scores$SampleID <- rownames(env_data) # Add SampleID column

In [122]:
if (inherits(env_data, "sample_data")) {
  env_data_df <- data.frame(sample_data(env_data))
} else {
  env_data_df <- env_data
}

# Merge scores with env_data
env_data_df$SampleID <- rownames(env_data_df)

# Now try merging with the converted data frame
merged_data <- merge(scores, env_data_df, by = "SampleID", all = FALSE)
print(dim(merged_data))

[1] 165  36


In [130]:
# Step 3: Fit Environmental Vectors Using envfit
envfit_results <- envfit(pcoa_results, env_data_df[, -1], permutations = 999, na.rm = TRUE)

# Extract significant vectors (p <= 0.05)
significant_envfit <- envfit_results$vectors$arrows[envfit_results$vectors$pvals <= 0.05, , drop = FALSE]
significant_envfit <- as.data.frame(significant_envfit)
significant_envfit$Variable <- rownames(significant_envfit)

# Print the results to verify
print(significant_envfit)

                                Dim1        Dim2                  Variable
Moisture                  -0.9770918 -0.21281827                  Moisture
Distance.from.previous.TH -0.3623307 -0.93204960 Distance.from.previous.TH
SA                        -0.7417656  0.67065921                        SA
Volume                    -0.9350985  0.35438793                    Volume
Depth                     -0.9980287 -0.06275838                     Depth
Dought_days                0.9754278  0.22031940               Dought_days


In [134]:
# Load necessary libraries
library(ape)
library(picante)

# Load the neighbor-joining tree
phylo_tree <- read.tree("../../3.2_Neutral_Stegen/figures/nj_tree_stegen.tre")

# Make sure the tree is rooted (if it's not already)
if (!is.rooted(phylo_tree)) {
  phylo_tree <- midpoint.root(phylo_tree)
}

# Now use this tree in your betaNTI calculation
beta_mntd <- comdistnt(as.matrix(otu_table), cophenetic(phylo_tree), abundance.weighted = TRUE)

# Calculate betaNTI
null_mntd <- ses.comdistnt(as.matrix(otu_table), cophenetic(phylo_tree), 
                           abundance.weighted = TRUE, runs = 999, null.model = "taxa.labels")
betaNTI <- null_mntd$obs.z

# Print the first few values of betaNTI to verify
print(head(betaNTI))

ERROR: Error in midpoint.root(phylo_tree): could not find function "midpoint.root"


In [137]:
phylo_tree


Phylogenetic tree with 3537 tips and 3535 internal nodes.

Tip labels:
  ASV_623, ASV_2415, ASV_3356, ASV_412, ASV_561, ASV_1809, ...

Unrooted; includes branch lengths.

In [146]:
# Load necessary libraries
library(ape)
library(picante)

# Load the neighbor-joining tree
phylo_tree <- read.tree("../../3.2_Neutral_Stegen/figures/nj_tree_stegen.tre")

# Make sure the tree is rooted (if it's not already)
if (!is.rooted(phylo_tree)) {
  phylo_tree <- root(phylo_tree, 1)  # Root at the first taxon
}

# Now use this tree in your betaNTI calculation
beta_mntd <- comdistnt(t(as.matrix(otu_table)), cophenetic(phylo_tree), abundance.weighted = TRUE)

# # Calculate betaNTI
# null_mntd <- ses.comdistnt(as.matrix(otu_table), cophenetic(phylo_tree), 
#                            abundance.weighted = TRUE, runs = 999, null.model = "taxa.labels")
# betaNTI <- null_mntd$obs.z

# # Print the first few values of betaNTI to verify
# print(head(betaNTI))

[1] "Dropping taxa from the community because they are not present in the distance matrix:"
[1] "ASV_2170" "ASV_250"  "ASV_2939"


ERROR: Error in dis[colnames(comm), colnames(comm)]: subscript out of bounds


In [147]:
# For the OTU table
dim(otu_table)
head(rownames(otu_table))

# For the phylogenetic tree
length(phylo_tree$tip.label)
head(phylo_tree$tip.label)

[1] 3540  165

[1] "ASV_1"    "ASV_10"   "ASV_100"  "ASV_1000" "ASV_1001" "ASV_1002"

[1] 3537

[1] "ASV_623"  "ASV_2415" "ASV_3356" "ASV_412"  "ASV_561"  "ASV_1809"

In [148]:
# Get the common ASVs
common_asvs <- intersect(rownames(otu_table), phylo_tree$tip.label)

# Subset the OTU table
otu_table_aligned <- otu_table[common_asvs, ]

# Prune the phylogenetic tree
phylo_tree_pruned <- keep.tip(phylo_tree, common_asvs)

# Recalculate the distance matrix
dist_matrix <- cophenetic(phylo_tree_pruned)

In [149]:
# Transpose the OTU table for ses.comdistnt function
otu_table_t <- t(otu_table_aligned)

# Calculate betaNTI
null_mntd <- ses.comdistnt(otu_table_t, dist_matrix, 
                           abundance.weighted = TRUE, runs = 999, null.model = "taxa.labels")
betaNTI <- null_mntd$obs.z

# Print the first few values of betaNTI to verify
print(head(betaNTI))

ERROR: Error in ses.comdistnt(otu_table_t, dist_matrix, abundance.weighted = TRUE, : could not find function "ses.comdistnt"


In [151]:
# Get the common ASVs
common_asvs <- intersect(rownames(otu_table), phylo_tree$tip.label)

# Subset the OTU table
otu_table_aligned <- otu_table[common_asvs, ]

# Prune the phylogenetic tree
phylo_tree_pruned <- keep.tip(phylo_tree, common_asvs)

# Recalculate the distance matrix
dist_matrix <- cophenetic(phylo_tree_pruned)

# Transpose the OTU table for ses.comdistnt function
otu_table_t <- t(otu_table_aligned)

# Calculate betaNTI
null_mntd <- ses.comdistnt(otu_table_t, dist_matrix, 
                           abundance.weighted = TRUE, runs = 999, null.model = "taxa.labels")
betaNTI <- null_mntd$obs.z

# Print the first few values of betaNTI to verify
print(head(betaNTI))

ERROR: Error in ses.comdistnt(otu_table_t, dist_matrix, abundance.weighted = TRUE, : could not find function "ses.comdistnt"


In [150]:
# Load necessary libraries
library(ape)
library(picante)

In [144]:
# For the community data
dim(otu_table)
head(colnames(otu_table))

# For the distance matrix
dim(cophenetic(phylo_tree))
head(rownames(cophenetic(phylo_tree)))

[1] 3540  165

[1] "F.D.DNA.P1.A3.A.01.ID3.3.15.THS.2.16S.3" 
[2] "F.D.DNA.P1.A3.A.02.ID3.3.15.THS.10.16S.3"
[3] "F.D.DNA.P1.A3.A.03.ID10.3.15.THS.9.16S.3"
[4] "F.D.DNA.P1.A3.A.04.ID7.7.15.THS.6.16S.3" 
[5] "F.D.DNA.P1.A3.A.05.ID10.2.15.THS.4.16S.3"
[6] "F.D.DNA.P1.A3.A.06.ID17.2.15.THS.1.16S.3"

[1] 3537 3537

[1] "ASV_623"  "ASV_2415" "ASV_3356" "ASV_412"  "ASV_561"  "ASV_1809"

In [142]:
as.matrix(otu_table)

,F.D.DNA.P1.A3.A.01.ID3.3.15.THS.2.16S.3,F.D.DNA.P1.A3.A.02.ID3.3.15.THS.10.16S.3,F.D.DNA.P1.A3.A.03.ID10.3.15.THS.9.16S.3,F.D.DNA.P1.A3.A.04.ID7.7.15.THS.6.16S.3,F.D.DNA.P1.A3.A.05.ID10.2.15.THS.4.16S.3,F.D.DNA.P1.A3.A.06.ID17.2.15.THS.1.16S.3,F.D.DNA.P1.A3.A.09.ID03.11.15.THS.6.16S.3,F.D.DNA.P1.A3.A.10.ID29.09.15.THS.4.16S.3,F.D.DNA.P1.A3.A.11.ID14.03.15.THS.1.16S.3,F.D.DNA.P1.A3.A.12.ID11.05.15.THS.1.16S.3,...,F.D.DNA.P3.C3.F.03.ID16.06.15.THS.3.16S.3,F.D.DNA.P3.C3.F.05.ID23.06.15.THS.9.16S.3,F.D.DNA.P3.C3.G.01.ID09.06.15.THS.7.16S.3,F.D.DNA.P3.C3.G.02.ID14.01.15.THS.6.16S.3,F.D.DNA.P3.C3.G.03.ID16.06.15.THS.4.16S.3,F.D.DNA.P3.C3.G.04.ID23.06.15.THS.1.16S.3,F.D.DNA.P3.C3.G.05.ID23.06.15.THS.10.16S.3,F.D.DNA.P3.C3.H.02.ID14.01.15.THS.7.16S.3,F.D.DNA.P3.C3.H.03.ID16.06.15.THS.5.16S.3,F.D.DNA.P3.C3.H.04.ID23.06.15.THS.3.16S.3
ASV_1,170,4,371,213,66,63,35,100,115,104,...,18,338,177,51,228,331,15,214,0,15
ASV_10,0,30,0,0,1,0,0,0,0,0,...,240,0,0,0,2,0,7,0,1,319
ASV_100,13,0,17,23,3,7,0,19,12,45,...,4,24,160,0,18,14,0,9,0,1
ASV_1000,2,0,8,0,1,1,0,0,3,1,...,0,2,17,0,0,1,0,20,0,0
ASV_1001,1,0,0,8,1,4,3,3,1,6,...,1,4,4,1,1,2,0,6,0,0
ASV_1002,0,0,1,0,1,3,0,1,1,3,...,0,0,29,0,0,0,0,13,0,0
ASV_1003,9,0,0,1,6,1,1,1,0,0,...,0,0,0,4,4,4,0,0,0,0
ASV_1004,3,1,2,0,0,0,0,1,0,0,...,1,0,0,3,2,4,4,1,2,3
ASV_1005,4,1,0,0,5,0,0,8,0,0,...,2,1,0,0,2,1,1,0,0,0
ASV_1006,2,6,0,0,6,0,0,0,0,0,...,2,0,0,2,11,1,2,0,0,2


In [141]:
cophenetic(phylo_tree)

,ASV_623,ASV_2415,ASV_3356,ASV_412,ASV_561,ASV_1809,ASV_2530,ASV_837,ASV_3460,ASV_642,...,ASV_465,ASV_69,ASV_1365,ASV_3296,ASV_2921,ASV_1196,ASV_1756,ASV_3368,ASV_1674,ASV_231
ASV_623,0.0000000,0.12992067,0.12521957,0.13331975,0.13366826,0.13357143,0.110664987,0.118822589,0.111341580,0.107035264,...,0.01599345,0.01288705,0.02120361,0.02120361,0.01015321,0.004352973,0.004352973,0.004260102,0.004292812,-1.928148e-05
ASV_2415,0.1299207,0.00000000,0.02603451,0.04267763,0.04302614,0.04292931,0.034200283,0.042357885,0.044359139,0.040052823,...,0.13857142,0.13546502,0.14378158,0.14378158,0.13273119,0.132973150,0.132973150,0.133872986,0.134252044,1.299400e-01
ASV_3356,0.1252196,0.02603451,0.00000000,0.03797654,0.03832505,0.03822821,0.029499185,0.037656787,0.039658041,0.035351726,...,0.13387033,0.13076392,0.13908048,0.13908048,0.12803009,0.128272052,0.128272052,0.129171888,0.129550947,1.252389e-01
ASV_412,0.1333198,0.04267763,0.03797654,0.00000000,0.00427353,0.01055110,0.037599365,0.045756968,0.047758221,0.043451906,...,0.14197051,0.13886410,0.14718066,0.14718066,0.13613027,0.136372232,0.136372232,0.137272068,0.137651127,1.333390e-01
ASV_561,0.1336683,0.04302614,0.03832505,0.00427353,0.00000000,0.01089961,0.037947876,0.046105478,0.048106732,0.043800416,...,0.14231902,0.13921261,0.14752917,0.14752917,0.13647878,0.136720743,0.136720743,0.137620579,0.137999638,1.336875e-01
ASV_1809,0.1335714,0.04292931,0.03822821,0.01055110,0.01089961,0.00000000,0.037851042,0.046008645,0.048009898,0.043703583,...,0.14222218,0.13911578,0.14743234,0.14743234,0.13638194,0.136623909,0.136623909,0.137523745,0.137902804,1.335907e-01
ASV_2530,0.1106650,0.03420028,0.02949918,0.03759937,0.03794788,0.03785104,0.000000000,0.008565481,0.025103456,0.020797140,...,0.11931574,0.11620933,0.12452590,0.12452590,0.11347550,0.113717467,0.113717467,0.114617303,0.114996361,1.106843e-01
ASV_837,0.1188226,0.04235789,0.03765679,0.04575697,0.04610548,0.04600864,0.008565481,0.000000000,0.033261058,0.028954743,...,0.12747334,0.12436694,0.13268350,0.13268350,0.12163310,0.121875069,0.121875069,0.122774905,0.123153964,1.188419e-01
ASV_3460,0.1113416,0.04435914,0.03965804,0.04775822,0.04810673,0.04800990,0.025103456,0.033261058,0.000000000,0.008560836,...,0.11999233,0.11688593,0.12520249,0.12520249,0.11415210,0.114394060,0.114394060,0.115293896,0.115672954,1.113609e-01
ASV_642,0.1070353,0.04005282,0.03535173,0.04345191,0.04380042,0.04370358,0.020797140,0.028954743,0.008560836,0.000000000,...,0.11568602,0.11257961,0.12089618,0.12089618,0.10984578,0.110087744,0.110087744,0.110987580,0.111366639,1.070545e-01


In [139]:
# Assuming 'dist_matrix' is your distance matrix and 'community_data' is your community data
common_taxa <- intersect(rownames(dist_matrix), colnames(community_data))
common_samples <- intersect(colnames(dist_matrix), rownames(community_data))

# Subset both datasets
dist_matrix_aligned <- dist_matrix[common_taxa, common_taxa]
community_data_aligned <- community_data[common_samples, common_taxa]

# Now proceed with your analysis using these aligned datasets

ERROR: Error: object 'dist_matrix' not found


In [140]:
# Assuming 'dist_matrix' is your distance matrix and 'community_data' is your community data
common_taxa <- intersect(rownames(dist_matrix), colnames(community_data))
common_samples <- intersect(colnames(dist_matrix), rownames(community_data))

# Subset both datasets
dist_matrix_aligned <- dist_matrix[common_taxa, common_taxa]
community_data_aligned <- community_data[common_samples, common_taxa]

# Now proceed with your analysis using these aligned datasets

ERROR: Error: object 'dist_matrix' not found


In [131]:
# Step 4: Calculate BetaNTI
# Compute betaMNTD (mean nearest taxon distance) between communities
beta_mntd <- comdistnt(as.matrix(otu_table), cophenetic(phylo_tree), abundance.weighted = TRUE)

# Generate null model and calculate betaNTI
beta_reps <- replicate(999, comdistnt(randomizeMatrix(otu_table, null.model = "taxa.labels"), 
                                      cophenetic(phylo_tree), abundance.weighted = TRUE))

# Calculate betaNTI
beta_mntd_mean <- apply(beta_reps, c(1,2), mean) # Mean betaMNTD from null model
beta_mntd_sd <- apply(beta_reps, c(1,2), sd) # Standard deviation of betaMNTD from null model
beta_nti <- (beta_mntd - beta_mntd_mean) / beta_mntd_sd # Calculate betaNTI

# Convert betaNTI to a long format for plotting
beta_nti_df <- as.data.frame(as.table(as.matrix(beta_nti))) # Convert betaNTI matrix to data frame
colnames(beta_nti_df) <- c("SampleID1", "SampleID2", "betaNTI") # Rename columns for better readability

# Step 5: Identify Significant BetaNTI Values
beta_nti_df$AssemblyProcess <- with(beta_nti_df, ifelse(beta_nti_df$betaNTI > 2, "Variable Selection",
                                            ifelse(beta_nti_df$betaNTI < -2, "Homogeneous Selection", "Stochastic Processes"))) # Categorize assembly processes based on betaNTI thresholds

ERROR: Error: object 'phylo_tree' not found


In [ ]:
# Step 6: Create the Ordination Plot with Environmental Vectors
p <- ggplot(scores, aes(x = PCoA1, y = PCoA2)) +
  geom_point(aes(color = env_variable_of_interest, shape = categorical_variable), size = 3) + # Plot samples with colors and shapes based on environmental and categorical variables
  scale_color_gradient(low = "blue", high = "red") + # Gradient color scale for environmental variable
  theme_minimal() +
  labs(x = paste("PCoA1 (", round(pcoa_results$eig[1]/sum(pcoa_results$eig)*100, 2), "%)", sep = ""),
       y = paste("PCoA2 (", round(pcoa_results$eig[2]/sum(pcoa_results$eig)*100, 2), "%)", sep = ""),
       color = "Environmental Variable",
       shape = "Categorical Variable") # Label axes and legends

# Add environmental vectors
p <- p + geom_segment(data = significant_envfit, aes(x = 0, y = 0, xend = PCoA1, yend = PCoA2),
                      arrow = arrow(length = unit(0.3, "cm")), color = "black") + # Add arrows for significant environmental vectors
  geom_text(data = significant_envfit, aes(x = PCoA1 * 1.1, y = PCoA2 * 1.1, label = Variable),
            size = 4, hjust = 0.5) # Add labels for environmental vectors

# Merge scores with betaNTI data
beta_nti_plot_data <- merge(beta_nti_df, scores[, c("SampleID", "PCoA1", "PCoA2")], 
                            by.x = "SampleID1", by.y = "SampleID", all.x = TRUE) # Merge betaNTI data with PCoA scores
beta_nti_plot_data <- merge(beta_nti_plot_data, scores[, c("SampleID", "PCoA1", "PCoA2")], 
                            by.x = "SampleID2", by.y = "SampleID", all.x = TRUE, suffixes = c(".1", ".2")) # Merge again for the second sample in each pair

# Filter for significant betaNTI values
beta_nti_significant <- beta_nti_plot_data[beta_nti_plot_data$AssemblyProcess != "Stochastic Processes", ] # Filter for significant betaNTI values

# Add lines to the plot
p <- p + geom_segment(data = beta_nti_significant,
                      aes(x = PCoA1.1, y = PCoA2.1, xend = PCoA1.2, yend = PCoA2.2,
                          linetype = AssemblyProcess),
                      color = "grey50", alpha = 0.5) # Add lines representing significant assembly processes

# Define linetype mapping
p <- p + scale_linetype_manual(values = c("Variable Selection" = "solid", 
                                          "Homogeneous Selection" = "dashed")) # Define line types for different assembly processes

# Update legend
p <- p + guides(linetype = guide_legend(title = "Assembly Process")) # Update legend to include line types

# Print the updated plot
print(p)

# Step 7: Create Heatmap of BetaNTI Values
beta_nti_matrix <- as.matrix(beta_nti) # Convert betaNTI to matrix

# Replace the diagonal and lower triangle with NA to avoid redundancy
beta_nti_matrix[lower.tri(beta_nti_matrix, diag = TRUE)] <- NA

# Create the heatmap
pheatmap(beta_nti_matrix,
         cluster_rows = TRUE, # Cluster rows
         cluster_cols = TRUE, # Cluster columns
         display_numbers = FALSE, # Do not display numbers
         color = colorRampPalette(c("blue", "white", "red"))(50), # Color gradient for heatmap
         na_col = "grey90", # Color for NA values
         main = "Heatmap of BetaNTI Values",
         fontsize = 8) # Set font size

# Step 8: Create Scatter Plot of Environmental Distance vs. BetaNTI
env_dist <- dist(env_data[, -1]) # Exclude SampleID column and calculate environmental distance
env_dist_df <- as.data.frame(as.table(as.matrix(env_dist))) # Convert distance matrix to data frame
colnames(env_dist_df) <- c("SampleID1", "SampleID2", "EnvDistance") # Rename columns for better readability

# Merge with betaNTI data
beta_nti_env <- merge(beta_nti_df, env_dist_df, by = c("SampleID1", "SampleID2")) # Merge environmental distance with betaNTI data

# Plot the scatter plot
ggplot(beta_nti_env, aes(x = EnvDistance, y = betaNTI)) +
  geom_point(alpha = 0.5) + # Plot points with transparency
  geom_smooth(method = "lm", color = "blue", se = TRUE) + # Add linear regression line with confidence interval
  theme_minimal() +
  labs(x = "Environmental Distance", y = "betaNTI",
       title = "Relationship Between Environmental Distance and betaNTI") + # Label axes and title
  stat_cor(method = "pearson", label.x = 0.1, label.y = max(beta_nti_env$betaNTI, na.rm = TRUE)) # Add Pearson correlation coefficient

# Step 9: Create Histogram of BetaNTI Distribution
ggplot(beta_nti_df, aes(x = betaNTI)) +
  geom_histogram(binwidth = 1, fill = "steelblue", color = "black") + # Plot histogram of betaNTI values
  geom_vline(xintercept = c(-2, 2), linetype = "dashed", color = "red") + # Add vertical lines for threshold values
  theme_minimal() +
  labs(x = "betaNTI", y = "Frequency",
       title = "Distribution of betaNTI Values") + # Label axes and title
  annotate("text", x = -3, y = Inf, label = "Homogeneous Selection", angle = 90, vjust = -0.5, color = "red") + # Annotate homogeneous selection
  annotate("text", x = 3, y = Inf, label = "Variable Selection", angle = 90, vjust = -0.5, color = "red") # Annotate variable selection
